In [1]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.01
batch_size=1
ngpu=1
ngf,nc = 3,3
ndf = 64
shape=(47021,10)

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_test.csv").sort_values(by=['date', 'warehouse'])

x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)).reshape((1, 47021, 9)))
x.shape #4315682

class EC_NET(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(9, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),
            
            nn.Linear(1524, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 424),
            nn.BatchNorm1d(424),
            nn.LeakyReLU(),
            
            nn.Linear(424, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),

            nn.Linear(1524, 10)
            
        )

    def forward(self,x):
        
        return self.rafire(x)
        
ec_net = EC_NET().type(torch.float32).to(device).eval()
ec_net= nn.DataParallel(ec_net).to(device)
#ec_net.load_state_dict(torch.load("/kaggle/input/encoder-weight-data/weight.pth",weights_only=False,map_location=torch.device('cpu')))

j=0
for i in x:
    print(f"loding batch : {j}")
    if j==0:
        encode = ec_net(i).cpu().detach().numpy().reshape(shape)
    else:
        encode = numpy.concatenate((encode, ec_net(i).cpu().detach().numpy().reshape(shape)), axis=0)
        
    #if j==25:
    #    break
    j+=1

x=torch.Tensor(encode)#[:3000]
len(x)

loding batch : 0


47021

In [5]:
y=pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_test.csv")
y=y[['unique_id', 'date']].to_numpy()

In [6]:
test_x = torch.utils.data.DataLoader(x,batch_size=batch_size)
test_x_ = torch.utils.data.DataLoader(y,batch_size=batch_size)
len(test_x)

47021

In [7]:


def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight.data, gain=0.02)
        
class EffnetModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rafire_1 = nn.LSTMCell(10, 1524)
        self.rafire_1_b = nn.Sequential(nn.BatchNorm1d(1524),
                                        nn.LeakyReLU())

        self.rafire_2 = nn.LSTMCell(1524, 1324)
        self.rafire_2_b = nn.Sequential(nn.BatchNorm1d(1324),
                                        nn.LeakyReLU())

        self.rafire_3 = nn.LSTMCell(1324, 1024)
        self.rafire_3_b = nn.Sequential(nn.BatchNorm1d(1024),
                                        nn.LeakyReLU())

        self.rafire_4 = nn.LSTMCell(1024, 824)
        self.rafire_4_b = nn.Sequential(nn.BatchNorm1d(824),
                                        nn.LeakyReLU())

        self.rafire_5 = nn.LSTMCell(824, 524)
        self.rafire_5_b = nn.Sequential(nn.BatchNorm1d(524),
                                        nn.LeakyReLU())

        self.rafire_6 = nn.LSTMCell(524, 324)
        self.rafire_6_b = nn.Sequential(nn.BatchNorm1d(324),
                                        nn.LeakyReLU())

        self.rafire_7 = nn.LSTMCell(324, 124)
        self.rafire_7_b = nn.Sequential(nn.BatchNorm1d(124),
                                        nn.LeakyReLU())
            
        self.rafire_8 = nn.Linear(124, 1)

    def forward(self,x):
        x,_=self.rafire_1(x)
        x=self.rafire_1_b(x)
        x,_=self.rafire_2(x)
        x=self.rafire_2_b(x)
        x,_=self.rafire_3(x)
        x=self.rafire_3_b(x)
        x,_=self.rafire_4(x)
        x=self.rafire_4_b(x)
        x,_=self.rafire_5(x)
        x=self.rafire_5_b(x)
        x,_=self.rafire_6(x)
        x=self.rafire_6_b(x)
        x,_=self.rafire_7(x)
        x=self.rafire_7_b(x)
        
        return self.rafire_8(x)

In [8]:
EFF_NET = EffnetModel().float().eval()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
EFF_NET.load_state_dict(torch.load("/kaggle/input/sales-forecasting/57.1866569519043.pth",weights_only=False,map_location=torch.device('cpu')))

<All keys matched successfully>

In [9]:
j=0
submission={'id' : [], 'sales_hat' : []}
for data in test_x:
    submission['id'].append(f"{str(y[j][0])}_{y[j][1]}")
    output = EFF_NET(data.to(device)).view(-1).detach().numpy()[0]
    submission['sales_hat'].append(output)
    j+=1
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)